In [ ]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
!pip install gcloud
!gcloud auth application-default login

# Import necessary Python libraries
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling
import re                          # Regular expressions

# Note: The actual imports remain exactly as in the original code

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.4/454.4 kB 13.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gcloud: filename=gcloud-0.18.3-py3-none-any.whl size=602927 sha256=bab9958b0a4264d22b74e2534cb7b140fea8b6e5110a0edbc50ae256dd63159f
  Stored in directory: /root/.cache/pip/wheels/2a/62/75/3d74209bfebb8805823ae74afa28653aa1ea76d8b5a9d741ff
Successfully built gcloud
Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=E6DeFnq0OsUGL9pISyyTs6rHN4TGWu&prompt=consent&token_usage=remote&access_type=offline&code_chal

# O Ano de 2024

In [ ]:
df = pd.read_excel('/content/Base_Estadic_2024.xlsx', sheet_name='Habitação', nrows=29, usecols=['Cod UF','Ehab01', 'Ehab03','Ehab04', 'Ehab05', 'Ehab06'])
df

,Cod UF,Ehab01,Ehab03,Ehab04,Ehab05,Ehab06
0,11,Setor subordinado a outra secretaria,Masculino,42,Parda,Especialização
1,12,Secretaria estadual em conjunto com outras pol...,Masculino,45,Branca,Ensino superior completo
2,13,Setor subordinado a outra secretaria,Masculino,31,Branca,Ensino superior completo
3,14,Órgão da administração indireta,Feminino,61,Branca,Especialização
4,15,Órgão da administração indireta,Masculino,54,Parda,Especialização
5,16,Secretaria estadual exclusiva,Feminino,55,Parda,Mestrado
6,17,Secretaria estadual em conjunto com outras pol...,Masculino,59,Parda,Ensino superior completo
7,21,Setor subordinado a outra secretaria,Masculino,46,Branca,Ensino superior completo
8,22,Órgão da administração indireta,Masculino,45,Parda,Ensino superior completo
9,23,Setor subordinado a outra secretaria,Masculino,69,Parda,Ensino superior completo


In [ ]:
uf = pd.read_excel('/content/Base_Estadic_2020.xlsx', sheet_name = 'Variáveis externas', usecols=[1,2,3])
uf

,Código da Unidade da Federação,Sigla da Unidade da Federação,Nome da Unidade da Federação
0,11,RO,Rondônia
1,12,AC,Acre
2,13,AM,Amazonas
3,14,RR,Roraima
4,15,PA,Pará
5,16,AP,Amapá
6,17,TO,Tocantins
7,21,MA,Maranhão
8,22,PI,Piauí
9,23,CE,Ceará


Renomeando as colunas

In [ ]:
df= df.rename(columns={'Cod UF':'cod_uf',
                       'Ehab01':'caracterizacao_orgao_gestor',
                        'Ehab03':'genero',
                        'Ehab04':'idade',
                        'Ehab05':'cor_raca',
                        'Ehab06':'grau_instrucao'})

In [ ]:
df['ano']=2024

In [ ]:
x= uf.pivot_table(columns=('Código da Unidade da Federação', 'Sigla da Unidade da Federação', 'Nome da Unidade da Federação'), aggfunc='size')


In [ ]:
uf = pd.DataFrame(x).reset_index()[['Código da Unidade da Federação', 'Sigla da Unidade da Federação', 'Nome da Unidade da Federação']]

In [ ]:
df = df.merge(uf, right_on='Código da Unidade da Federação',left_on='cod_uf') #juntando os dataframes, adicionando sigla e nome das UFs


In [ ]:
df = df.drop(['Código da Unidade da Federação'], axis=1) #eliminando coluna repetida

In [ ]:
df = df.rename(columns={'Sigla da Unidade da Federação':'sigla_uf',
                        'Nome da Unidade da Federação':'uf'}) #padronizando as colunas

In [ ]:
df['idade']=np.where(df['idade']=='Não informou',np.nan,df['idade'])
df['idade'] =pd.to_numeric(df['idade'])


In [ ]:
limites = [0, 30, 50,65,100]
categorias = ['Entre 18-29', 'Entre 30-49', 'Entre 50-64', 'Acima de 65']

df['faixa_etaria'] = pd.cut(df['idade'], bins=limites, labels=categorias)


In [ ]:
df.columns

Index(['cod_uf', 'caracterizacao_orgao_gestor', 'genero', 'idade', 'cor_raca',
       'grau_instrucao', 'ano', 'sigla_uf', 'uf', 'faixa_etaria'],
      dtype='object')

In [ ]:
df= df[['ano', 'sigla_uf','cod_uf', 'uf', 'caracterizacao_orgao_gestor','genero', 'faixa_etaria', 'cor_raca','grau_instrucao']]

In [ ]:
df['grau_instrucao'].unique()

array(['Especialização', 'Ensino superior completo', 'Mestrado',
       'Ensino superior incompleto'], dtype=object)

In [ ]:
# criando dicionário
dict_esco = {'Ensino médio (2º Grau) completo':'Até Ensino Médio',
             'Ensino superior completo':'Até Ensino Superior',
             'Ensino superior incompleto':'Até Ensino Superior',
             'Especialização':'Até Pós Graduação ou Mestrado',
             'Mestrado':'Até Pós Graduação ou Mestrado',
             'Doutorado':'Até Doutorado'}


In [ ]:
df = df.replace({'grau_instrucao':dict_esco})

In [ ]:
df['grau_instrucao'].unique()

array(['Até Pós Graduação ou Mestrado', 'Até Ensino Superior Completo'],
      dtype=object)

In [ ]:
df.columns

Index(['ano', 'sigla_uf', 'cod_uf', 'uf', 'caracterizacao_orgao_gestor',
       'genero', 'faixa_etaria', 'cor_raca', 'grau_instrucao'],
      dtype='object')

In [ ]:
df

,ano,sigla_uf,cod_uf,uf,caracterizacao_orgao_gestor,genero,faixa_etaria,cor_raca,grau_instrucao
0,2020,RO,11,Rondônia,Setor subordinado a outra secretaria,Feminino,Entre 30-49,Parda,Até Ensino Superior Completo
1,2020,AC,12,Acre,Setor subordinado a outra secretaria,Masculino,Entre 30-49,Parda,Até Ensino Superior Completo
2,2020,AM,13,Amazonas,Órgão da administração indireta,Masculino,Entre 50-64,Branca,Até Ensino Superior Completo
3,2020,RR,14,Roraima,Órgão da administração indireta,Masculino,Entre 50-64,Parda,Até Ensino Superior Completo
4,2020,PA,15,Pará,Órgão da administração indireta,Masculino,Entre 30-49,Parda,Até Ensino Superior Completo
5,2020,AP,16,Amapá,Secretaria estadual em conjunto com outras pol...,Masculino,Entre 30-49,Parda,Até Pós Graduação ou Mestrado
6,2020,TO,17,Tocantins,Secretaria estadual em conjunto com outras pol...,Masculino,Entre 30-49,Branca,Até Pós Graduação ou Mestrado
7,2020,MA,21,Maranhão,Secretaria estadual exclusiva,Masculino,Entre 30-49,Branca,Até Pós Graduação ou Mestrado
8,2020,PI,22,Piauí,Setor subordinado diretamente à chefia do Exec...,Feminino,Entre 50-64,Parda,Até Ensino Superior Completo
9,2020,CE,23,Ceará,Setor subordinado a outra secretaria,Masculino,Acima de 65,Parda,Até Ensino Médio


# Consumindo a base anterior para agregar o novo ano

In [ ]:


query = """SELECT * FROM `repositoriodedadosgpsp.cargos_lideranca.ESTADIC_perfil_gestor_habitacao_tipo_orgao`"""
# Execute the query using pandas_gbq.read_gbq and load the result into a pandas DataFrame called 'df'.
# The 'project_id' specifies the Google Cloud Project to use.
df_old = pandas_gbq.read_gbq(query, project_id='repositoriodedadosgpsp')



Downloading: 100%|██████████|


In [ ]:
df_final = pd.concat([df, df_old], ignore_index=True)

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ano                          54 non-null     Int64  
 1   sigla_uf                     54 non-null     object 
 2   cod_uf                       54 non-null     Int64  
 3   uf                           54 non-null     object 
 4   caracterizacao_orgao_gestor  54 non-null     object 
 5   genero                       54 non-null     object 
 6   faixa_etaria                 54 non-null     object 
 7   cor_raca                     54 non-null     object 
 8   grau_instrucao               54 non-null     object 
 9   idade                        27 non-null     float64
dtypes: Int64(2), float64(1), object(7)
memory usage: 4.5+ KB


In [ ]:
df_final['grau_instrucao'].unique()

array(['Até Pós Graduação ou Mestrado', 'Até Ensino Superior Completo',
       'Até Ensino Médio', 'Até Doutorado'], dtype=object)

In [ ]:
df_final['grau_instrucao']=np.where(df_final['grau_instrucao']=='Até Ensino Superior Completo','Até Ensino Superior',df_final['grau_instrucao'])

In [ ]:
df_final['grau_instrucao'].unique()

array(['Até Pós Graduação ou Mestrado', 'Até Ensino Superior',
       'Até Ensino Médio', 'Até Doutorado'], dtype=object)

Subindo para o GBQ

In [ ]:
# Define the BigQuery table schema with Portuguese descriptions
schema=[bigquery.SchemaField('ano','INTEGER',description='Ano da apuração daquele dado'),
        bigquery.SchemaField('sigla_uf','STRING',description='sigla da UF'),
        bigquery.SchemaField('cod_uf','INTEGER',description='Código do IBGE da UF'),
        bigquery.SchemaField('uf','STRING',description='Nome da UF'),
        bigquery.SchemaField('caracterizacao_orgao_gestor','STRING',description='Caracterização do órgão no qual o gestor está'),
        bigquery.SchemaField('genero','STRING',description='Gênero autodeclarado ou não'),
        bigquery.SchemaField('faixa_etaria','STRING',description='faixa etária da observação'),
        bigquery.SchemaField('cor_raca','STRING',description='Raça/cor da pessoa observada'),
        bigquery.SchemaField('grau_instrucao','STRING',description='Escolaridade da pessoa ou do vínculo observado com detalhamento na pós-graduação')
        ]

# Initialize BigQuery client connection
client = bigquery.Client(project='repositoriodedadosgpsp')

# Create reference to target dataset
dataset_ref = client.dataset('cargos_lideranca')

# Create reference to target table with standardized naming convention:
# FONTE_algo_intuitivo_dado (MUNIC_quantidade_vinculos_mapa_v1)
table_ref = dataset_ref.table('ESTADIC_perfil_gestor_habitacao_tipo_orgao_v1')

# Configure the load job with our schema definition
job_config = bigquery.LoadJobConfig(
    schema=schema,
    # Optional parameters (commented out):
    # write_disposition="WRITE_TRUNCATE",  # Overwrites table if exists
    # create_disposition="CREATE_IF_NEEDED"  # Default behavior
)

# Execute the load job to upload DataFrame to BigQuery
job = client.load_table_from_dataframe(
    dataframe=df_final,
    destination=table_ref,
    job_config=job_config
)

# Wait for the job to complete
job.result()

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


LoadJob<project=repositoriodedadosgpsp, location=US, id=c11974af-26b7-47df-b6cc-d11e01ac0513>